# Multiple Linear Regression — Income Prediction

This notebook predicts a person's **income** based on their **age** and **experience**.

We train **5 different machine learning models**, compare their accuracy, and pick the best one.

---

## Workflow

1. **Load data** — read the CSV file
2. **Explore data** — check shape, columns, sample rows
3. **Split data** — separate features (X) from target (y), then split into train/test
4. **Train models** — fit 5 different ML algorithms
5. **Evaluate** — measure accuracy with R², MAE, MSE
6. **Save models** — store trained models as `.pkl` files for reuse
7. **Compare visually** — plot predictions and R² scores
8. **Make new predictions** — load the best model and predict on new data

## Step 1: Import libraries

We need four main libraries:

| Library | What it does |
|---------|-------------|
| **pandas** (`pd`) | Loads and handles tabular data (like Excel for Python) |
| **numpy** (`np`) | Math and array operations |
| **matplotlib** (`plt`) | Creates plots and charts |
| **seaborn** (`sns`) | Prettier statistical plots, built on matplotlib |

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Step 2: Load the dataset

The dataset is a CSV file with **3 columns**: `age`, `experience`, `income`.

- `r"..."` → a **raw string**. Use this for Windows file paths so backslashes work correctly.
- `pd.read_csv()` → reads the CSV into a **DataFrame** (pandas' table object).

👉 **Change the path** below to wherever you saved the CSV.

In [ ]:
df = pd.read_csv("multiple_linear_regression_dataset.csv")

## Step 3: Explore the data

Before training, always look at the data first to understand what you're working with.

- `df.head(2)` → shows the first 2 rows
- `df.shape` → tells us `(rows, columns)`
- `df.columns` → list of column names

Expected output: 20 rows × 3 columns (`age`, `experience`, `income`).

In [ ]:
df.head(2)

In [ ]:
print("Columns:", df.columns.tolist())
print("Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

## Step 4: Split into features (X) and target (y)

In machine learning:
- **X** = features (the inputs we use to predict) → `age`, `experience`
- **y** = target (what we want to predict) → `income`

We then split the data into:
- **Training set (80%)** — used to teach the model
- **Test set (20%)** — held back to check how well the model learned

💡 **`random_state=42`** makes the random split reproducible — you'll get the same split every time.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('income', axis=1)   # features: age, experience
y = df['income']                # target: income

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set:     {X_test.shape[0]} rows")

## Step 5: First model — K-Nearest Neighbors (KNN)

Let's start with one model to see the workflow before training all 5.

**KNN** predicts by looking at the **K closest data points** (default K=5) and averaging their values.

Three core steps for any model:
1. **Create** the model: `knn = KNeighborsRegressor()`
2. **Fit** (train) on training data: `knn.fit(X_train, y_train)`
3. **Predict** on new data: `knn.predict(X_test)`

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor()
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
print("First 10 predictions:", y_pred[:10])

## Step 6: Evaluate the model

How do we know if the model is any good? We compare its predictions (`y_pred`) to the actual values (`y_test`) using **3 standard regression metrics**:

| Metric | What it measures | Better when |
|--------|------------------|-------------|
| **MSE** (Mean Squared Error) | Average squared error. Punishes big mistakes more. | Lower ⬇️ |
| **MAE** (Mean Absolute Error) | Average absolute error. Easy to read — same units as target. | Lower ⬇️ |
| **R² Score** | How much variance the model explains. 1.0 = perfect, 0 = useless, negative = worse than guessing the average. | Higher ⬆️ |

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rs  = r2_score(y_test, y_pred)

print(f"Mean Squared Error:  {mse:.2f}")
print(f"Mean Absolute Error: {mae:.2f}")
print(f"R² Score:            {rs:.3f}")

## Step 7: Train 5 different models

Different algorithms work differently. We'll train all 5 and compare them to find the best one for this dataset.

| Short name | Full name | How it works |
|------------|-----------|--------------|
| **dt** | Decision Tree | Splits data into branches based on rules (like a flowchart) |
| **rf** | Random Forest | Many decision trees combined → more stable predictions |
| **svr** | Support Vector Regression | Fits a margin around the data |
| **gnb** | Gaussian Naive Bayes | Probability-based (note: usually for classification) |
| **lr** | Linear Regression | Fits a straight line through the data ⭐ |

We store them in a dictionary so we can loop through them easily.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LinearRegression

mdls = {
    'dt':  DecisionTreeRegressor(),
    'rf':  RandomForestRegressor(),
    'svr': SVR(),
    'gnb': GaussianNB(),
    'lr':  LinearRegression()
}

## Step 8: Train, evaluate, and save all models

We loop through every model and:
1. **Train** it on the training data → `model.fit(X_train, y_train)`
2. **Predict** on test data → `model.predict(X_test)`
3. **Calculate R²** → `r2_score(y_test, y_pred)`
4. **Save the model** to a `.pkl` file → `joblib.dump(model, ...)`

💾 **What is `.pkl`?** It's a saved trained model. You can load it later with `joblib.load("name.pkl")` and use it without retraining.

⚠️ **Naming tip**: We use `score_list` (not `r2_scores`) to avoid clashing with the `r2_score` function name.

In [ ]:
import joblib
from sklearn.metrics import r2_score

score_list = []   # store R² scores for plotting later

for name, model in mdls.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rs = r2_score(y_test, y_pred)
    score_list.append(rs)
    print(f"{name}: R² = {rs:.3f}")
    joblib.dump(model, f"{name}.pkl")     # save model to file

## Step 9: Visualize R² scores — line plot

A quick line plot showing R² for each model. The **highest point = best model**.

In [ ]:
mdl_name = list(mdls.keys())

plt.plot(mdl_name, score_list, marker='D')
plt.title("R² Score by Model")
plt.xlabel("Model")
plt.ylabel("R² Score")
plt.grid(True)
plt.show()

## Step 10: Visualize R² scores — bar chart

A clearer view: **green bars** are good models (R² > 0), **red bars** are bad ones (R² < 0).

A negative R² means the model performs **worse than just guessing the average**.

In [ ]:
# Train all models and collect scores
scores = {}
for name, model in mdls.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    scores[name] = r2_score(y_test, y_pred)

# Bar chart
plt.figure(figsize=(10, 6))
colors = ['green' if s > 0 else 'red' for s in scores.values()]
plt.bar(scores.keys(), scores.values(), color=colors)
plt.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
plt.ylabel('R² Score')
plt.title('Model Comparison — R² Score')
plt.xticks(rotation=45)

# Add value labels on top of bars
for i, (name, score) in enumerate(scores.items()):
    plt.text(i, score, f'{score:.2f}',
             ha='center',
             va='bottom' if score > 0 else 'top')

plt.tight_layout()
plt.show()

## Step 11: Load the best saved model

Earlier we saved each model as a `.pkl` file. Now we load the best one (**Linear Regression — `lr.pkl`**) to make new predictions.

📌 **Why this is useful in the real world**: in production, you train your model **once** and save it. Then your app loads the saved file and serves predictions instantly — no need to retrain.

In [ ]:
model = joblib.load('lr.pkl')
model

## Step 12: Make predictions on the full dataset

Now we use the loaded model to predict income for every row in `df`, then check the R² score against the actual incomes.

In [ ]:
income_pred = model.predict(df.drop('income', axis=1))
rs = r2_score(df['income'], income_pred)

print(f"R² on full dataset: {rs:.3f}")

## Step 13: Test the model on 20 random samples

We repeatedly grab a random sample of 20 rows and predict on them. This shows how stable the model is across different subsets of data.

If R² stays consistent across runs → the model is reliable. If it varies wildly → the model is unstable.

In [ ]:
for x in range(20):
    sample = df.sample(n=20)
    y_pred = model.predict(sample.drop('income', axis=1))
    rs = r2_score(sample['income'], y_pred)
    print(f"Run {x+1:2d}: R² = {rs:.3f}")

## Step 14: Visualize predictions — Experience vs Income

A scatter plot to *see* how well the model performs:

- 🔵 **Blue dots** = actual income values
- ❌ **Red X marks** = predicted income values

When red Xs land close to blue dots → model is accurate. When they're far apart → model is missing the pattern.

In [1]:
plt.figure(figsize=(8, 6))
plt.scatter(df['experience'], df['income'],
            color='blue', label='Actual data')
plt.scatter(df['experience'], model.predict(df.drop('income', axis=1)),
            color='red', label='Predicted', marker='x')
plt.xlabel('Experience')
plt.ylabel('Income')
plt.title('Experience vs Income')
plt.legend()
plt.show()

NameError: name 'plt' is not defined

## Summary

### What we did
1. ✅ Loaded a small income dataset (20 rows × 3 columns)
2. ✅ Split into 80% training / 20% testing
3. ✅ Trained **5 different models**: Decision Tree, Random Forest, SVR, Naive Bayes, Linear Regression
4. ✅ Compared them using **R² score**
5. ✅ Saved each model as a `.pkl` file
6. ✅ Loaded the best model and made predictions
7. ✅ Visualized results with line plot, bar chart, and scatter plot

### Key findings
- 🥇 **Linear Regression won** — best R² score → makes sense since income vs experience is a fairly linear relationship
- ❌ **Naive Bayes failed** — it's a classification algorithm, not meant for predicting numbers
- ❌ **SVR failed** — needs feature scaling to work properly
- 🌲 **Random Forest & Decision Tree** were okay but limited by the small dataset (only 20 rows)

### Takeaways
> **Match the algorithm to the problem.** Linear data → linear model. Don't always reach for the fanciest algorithm.

> **More data = better models.** With only 20 rows, complex models can't shine. Most algorithms need 100s or 1000s of rows.

### Next steps to try
- Scale features with `StandardScaler` and re-test SVR
- Use `cross_val_score` for a more reliable R² estimate on small data
- Try hyperparameter tuning with `GridSearchCV`
- Drop Naive Bayes (wrong tool for regression)